In [0]:
import pyspark.sql.functions as F
import requests

In [0]:
# Extract JSON
national = requests.get("https://api.census.gov/data/2020/dec/pl?get=GEO_ID,NAME,P1_001N,P1_003N,P1_004N,P1_005N,P1_006N,P1_007N,P1_008N,P1_009N,P2_001N,P2_002N,H1_001N,H1_002N&ucgid=0100000US").json()
national_columns = national[0]
national_data = national[1:]

county = requests.get("https://api.census.gov/data/2020/dec/pl?get=GEO_ID,NAME,P1_001N,P1_003N,P1_004N,P1_005N,P1_006N,P1_007N,P1_008N,P1_009N,P2_001N,P2_002N,H1_001N,H1_002N&for=county").json()
county_columns = county[0]
county_data = county[1:]

state = requests.get("https://api.census.gov/data/2020/dec/pl?get=GEO_ID,NAME,P1_001N,P1_003N,P1_004N,P1_005N,P1_006N,P1_007N,P1_008N,P1_009N,P2_001N,P2_002N,H1_001N,H1_002N&for=state").json()
state_columns = state[0]
state_data = state[1:]
   
msa = requests.get("https://api.census.gov/data/2020/dec/pl?get=GEO_ID,NAME,P1_001N,P1_003N,P1_004N,P1_005N,P1_006N,P1_007N,P1_008N,P1_009N,P2_001N,P2_002N,H1_001N,H1_002N&ucgid=pseudo(0100000US$3100000)").json()
msa_columns = msa[0]
msa_data = msa[1:]

# Create DataFrames
national_df = spark.createDataFrame(national_data, national_columns)
county_df = spark.createDataFrame(county_data, county_columns)
state_df = spark.createDataFrame(state_data, state_columns)
msa_df = spark.createDataFrame(msa_data, msa_columns)

# Specify the geo_level for each dataframe
national_df = national_df.withColumn('geo_level', F.lit('national'))
county_df = county_df.withColumn('geo_level', F.lit('county'))
state_df = state_df.withColumn('geo_level', F.lit('state'))
msa_df = msa_df.withColumn('geo_level', F.lit('msa'))

In [0]:
len(state_df.columns)

In [0]:
len(county_df.columns)

In [0]:
diff = set(county_df.columns) - set(state_df.columns)

print(diff)

In [0]:
display(county_df)

In [0]:
endpoint = "https://api.census.gov/data/2020/dec/pl?get=GEO_ID,NAME,P1_001N,P1_003N,P1_004N,P1_005N,P1_006N,P1_007N,P1_008N,P1_009N,P2_001N,P2_002N,H1_001N,H1_002N&for=state"

response = requests.get(endpoint)
if response.status_code == 200:
    json_data = response.json()
else:
    raise Exception(f"Request failed with status code {response.status_code}")

columns = json_data[0]
data = json_data[1:]

spark_df = spark.createDataFrame(data, columns)
spark_df = spark_df.withColumn('geo_level', F.lit('county'))
display(spark_df)

In [0]:
display(spark_df)